In [1]:
%mamba install scikit-learn matplotlib seaborn

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, scikit-learn, seaborn
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 1.884 seconds
  Name                Version     Build                Channel
----------------------------------------------------------------------------
+ ca-certificates     2026.11.20  hbd8a1cb_0           conda-forge
+ certifi             2026.11.20  pyhd8ed1ab_0         conda-forge
+ charset-normalizer  3.5.1       pyhd8ed1ab_0         conda-forge
+ idna                3.20        pyh5ded981_0         conda-forge
+ joblib              1.6.0       py313h57e9779_0      emscripten-forge-4x
+ narwhals            2.29.0      pyh5ded981_0         conda-forge
+ pandas              3.1.0       np23py313h124dbc7_0  emscripten-forge-4x
+ patsy               1.0.3       py313h1804a44_0      emscripten-forge-4x
+ requests            2.34.2      pyhcf101f3_0         conda-forge
+ scikit-learn  

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay
)

# Load shared train and test datasets from your GitHub repository
train_path = "data/train.csv" if os.path.exists("data/train.csv") else "train.csv"
test_path = "data/test.csv" if os.path.exists("data/test.csv") else "test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Train Shape: {train_df.shape} | Test Shape: {test_df.shape}")
print("\nColumns available:\n", train_df.columns.tolist())
train_df.head()

Train Shape: (960, 40) | Test Shape: (240, 40)

Columns available:
 ['customer_id', 'age', 'annual_income', 'credit_score', 'loan_amount', 'debt_to_income_ratio', 'late_payments_12m', 'account_tenure_months', 'employment_status', 'loan_default', 'customer_remark', 'slm_sentiment_label', 'slm_sentiment_score', 'slm_risk_category', 'slm_strain_score', 'slm_thinking_sentiment_label', 'slm_thinking_sentiment_score', 'slm_thinking_risk_category', 'slm_thinking_strain_score', 'customer_segment', 'loan_to_income_ratio', 'monthly_income', 'estimated_monthly_debt', 'estimated_monthly_installment', 'installment_to_income_ratio', 'total_burden_ratio', 'residual_monthly_income', 'late_payment_intensity', 'has_late_payments', 'chronic_delinquency_flag', 'credit_utilization_stress', 'risk_adjusted_leverage', 'log_annual_income', 'log_loan_amount', 'log_tenure_months', 'credit_score_tier', 'employment_stability_score', 'emp_Full-Time', 'emp_Part-Time', 'emp_Self-Employed']


,customer_id,age,annual_income,credit_score,loan_amount,debt_to_income_ratio,late_payments_12m,account_tenure_months,employment_status,loan_default,...,credit_utilization_stress,risk_adjusted_leverage,log_annual_income,log_loan_amount,log_tenure_months,credit_score_tier,employment_stability_score,emp_Full-Time,emp_Part-Time,emp_Self-Employed
0,CUST_10503,53,47185.21,610,27052.44,0.297,1,112,Full-Time,0,...,0.413852,0.237272,10.761857,10.205569,4.727388,1,3,1,0,0
1,CUST_10715,28,87768.42,785,29901.85,0.179,1,63,Full-Time,0,...,0.193822,0.066033,11.382468,10.305709,4.158883,3,3,1,0,0
2,CUST_11185,41,70426.91,708,15698.97,0.306,0,48,Full-Time,0,...,0.367373,0.081892,11.162345,9.661414,3.891820,2,3,1,0,0
3,CUST_10476,34,27546.41,630,12789.31,0.161,1,84,Full-Time,0,...,0.217222,0.100852,10.223664,9.456443,4.442651,1,3,1,0,0
4,CUST_11165,26,55309.27,558,24328.73,0.267,4,57,Part-Time,1,...,0.406720,0.178903,10.920714,10.099454,4.060443,0,0,0,1,0


In [3]:
print("=== Missing Values (Train) ===")
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])

print("\n=== Target Distribution (loan_default) ===")
print("Train:\n", train_df["loan_default"].value_counts(normalize=True).round(4))
print("Test:\n", test_df["loan_default"].value_counts(normalize=True).round(4))

=== Missing Values (Train) ===
Series([], dtype: int64)

=== Target Distribution (loan_default) ===
Train:
 loan_default
0    0.7938
1    0.2062
Name: proportion, dtype: float64
Test:
 loan_default
0    0.7917
1    0.2083
Name: proportion, dtype: float64


In [4]:
target_col = "loan_default"
exclude_cols = ["customer_id", "customer_remark", target_col]

# All valid predictor features (Tabular + Qwen3.5 SLM features)
all_feature_cols = [c for c in train_df.columns if c not in exclude_cols]

# Tabular-only features (excluding Qwen SLM columns & customer_segment)
slm_cols = [c for c in all_feature_cols if c.startswith("slm_") or c == "customer_segment"]
tabular_feature_cols = [c for c in all_feature_cols if c not in slm_cols]

print(f"Tabular-only features ({len(tabular_feature_cols)}):", tabular_feature_cols)
print(f"Qwen SLM features ({len(slm_cols)}):", slm_cols)

y_train = train_df[target_col]
y_test = test_df[target_col]

Tabular-only features (28): ['age', 'annual_income', 'credit_score', 'loan_amount', 'debt_to_income_ratio', 'late_payments_12m', 'account_tenure_months', 'employment_status', 'loan_to_income_ratio', 'monthly_income', 'estimated_monthly_debt', 'estimated_monthly_installment', 'installment_to_income_ratio', 'total_burden_ratio', 'residual_monthly_income', 'late_payment_intensity', 'has_late_payments', 'chronic_delinquency_flag', 'credit_utilization_stress', 'risk_adjusted_leverage', 'log_annual_income', 'log_loan_amount', 'log_tenure_months', 'credit_score_tier', 'employment_stability_score', 'emp_Full-Time', 'emp_Part-Time', 'emp_Self-Employed']
Qwen SLM features (9): ['slm_sentiment_label', 'slm_sentiment_score', 'slm_risk_category', 'slm_strain_score', 'slm_thinking_sentiment_label', 'slm_thinking_sentiment_score', 'slm_thinking_risk_category', 'slm_thinking_strain_score', 'customer_segment']
